# Final Assignment - Question 5

In [ ]:
!pip install yfinance beautifulsoup4 lxml plotly pandas requests -q

In [ ]:
import yfinance as yf
import pandas as pd
import requests
from bs4 import BeautifulSoup
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
def make_graph(stock_data, revenue_data, stock):
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=("Historical Share Price", "Historical Revenue"),
                        vertical_spacing=.3)
    stock_data_specific = stock_data[pd.to_datetime(stock_data.Date) <= pd.to_datetime("2021-06-14")]
    revenue_data_specific = revenue_data[pd.to_datetime(revenue_data.Date) <= pd.to_datetime("2021-04-30")]
    fig.add_trace(go.Scatter(x=pd.to_datetime(stock_data_specific.Date),
                             y=stock_data_specific.Close.astype("float"), name="Share Price"), row=1, col=1)
    fig.add_trace(go.Scatter(x=pd.to_datetime(revenue_data_specific.Date),
                             y=revenue_data_specific.Revenue.astype("float"), name="Revenue"), row=2, col=1)
    fig.update_xaxes(title_text="Date", row=1, col=1)
    fig.update_xaxes(title_text="Date", row=2, col=1)
    fig.update_yaxes(title_text="Price ($US)", row=1, col=1)
    fig.update_yaxes(title_text="Revenue ($US Millions)", row=2, col=1)
    fig.update_layout(showlegend=False, height=900, title=stock, xaxis_rangeslider_visible=True)
    fig.show()

## Question 1: Tesla stock data (yfinance)

In [ ]:
tesla = yf.Ticker("TSLA")
tesla_data = tesla.history(period="max")
tesla_data.reset_index(inplace=True)
tesla_data["Date"] = pd.to_datetime(tesla_data["Date"]).dt.tz_localize(None)
tesla_data.head()

## Question 2: Tesla revenue (web scraping)

In [ ]:
def scrape_revenue(url):
    soup = BeautifulSoup(requests.get(url).text, "html.parser")
    rows = []
    for tr in soup.find_all("tbody")[1].find_all("tr"):
        td = tr.find_all("td")
        rows.append({"Date": td[0].text, "Revenue": td[1].text.replace("$", "").replace(",", "")})
    df = pd.DataFrame(rows)
    df = df[df["Revenue"] != ""]
    df.dropna(inplace=True)
    return df

url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/revenue.htm"
tesla_revenue = scrape_revenue(url)
tesla_revenue.tail()

## Question 5: Tesla dashboard

In [ ]:
make_graph(tesla_data, tesla_revenue, "Tesla")